In [ ]:
import os
import numpy as np
import pandas as pd

from dekef.base_density import *
from dekef.kernel_function import *
from dekef.data_median_dist import *
from dekef.negloglik_params import *

from dekef.negloglik_grid_points import *
from dekef.scorematching_penalized_grid_points import *

from dekef.plot_density_1d import *

from IPython.display import Markdown as md

import warnings
warnings.filterwarnings('ignore')
np.random.seed(1989)

In [ ]:
os.chdir('../data')
df = np.load('geyser.npy').astype(np.float64)
waiting = df[:, 0].reshape(-1, 1)
waiting = waiting[waiting != 108.0]
contam_pt = 120.
waiting = np.append(waiting, contam_pt)

bw = 7.
kernel_type = 'gaussian_poly2'

In [ ]:
# set plot parameters
font_size = 15
xlimit = (1., 310.)
plot_pts_cnt = 3000
ylimit = (-0.001, 0.0701)

# plot the histogram of waiting variable 
ax = pd.Series(waiting.flatten()).hist(grid = False, figsize = (10, 10), bins = 'fd', density = True)
ax.set_xlim(xlimit)
ax.set_ylim(ylimit)
ax.set_title('Histogram of Waiting Variable', fontsize = font_size)
ax.set_xlabel('waiting', fontsize = font_size)
ax.set_ylabel('density', fontsize = font_size)
ax.tick_params(axis = 'both', labelsize = font_size)

In [ ]:
base_density = BasedenGamma(np.load('../data/geyser.npy').astype(np.float64)[:, 0])

In [ ]:
grid_points = np.arange(1., 311., 1).reshape(-1, 1)
sm_pen = ScoreMatchingPenalizedGridPoints(
    data = waiting, 
    grid_points = grid_points,
    base_density = base_density,
    kernel_type = 'gaussian_poly2', 
    kernel_r1 = 1.0, 
    kernel_r2 = 0.0, 
    kernel_c = 0.0, 
    kernel_bw = bw)

In [ ]:
log_penalty_param_list = np.arange(-15., 0., 0.5)

# specify the plotting parameters 
plot_params = plot_density_1d_params(xlimit, ylimit, plot_pts_cnt=plot_pts_cnt)

new_data = np.linspace(xlimit[0], xlimit[1], plot_pts_cnt)

sm_folder = (f'../data/PenSM-FinKEF-ContamData={contam_pt}-bw={bw}-kernel={kernel_type}-' + 
             f'plotdomain={xlimit}-plotcnts={plot_pts_cnt}')

if not os.path.isdir(sm_folder):
    os.mkdir(sm_folder)

file_name_newdata = f'/new_data.npy'
np.save(sm_folder + file_name_newdata, new_data)

file_name_grid_points = f'/grid_points.npy'
np.save(sm_folder + file_name_grid_points, grid_points)

In [ ]:
for log_pen_param in log_penalty_param_list: 
    
    print('*' * 50)
    print(f'log penalty = {log_pen_param}')
    
    coef_smpen = sm_pen.coef(
        data = waiting, 
        lambda_param = np.exp(log_pen_param))
    
    # visualize density estimate
    den_vals = plot_density_1d(
        data = waiting.reshape(-1, 1), 
        kernel_function = sm_pen.kernel_function_gridpoints, 
        base_density = base_density, 
        basis_type = 'grid_points', 
        coef = coef_smpen, 
        normalizing = True, 
        method = 'penalized score matching', 
        x_label = 'waiting',
        grid_points = grid_points,
        save_plot = False, 
        save_dir = None, 
        save_filename = None, 
        plot_kwargs = plot_params)
    
    file_name_coef = f'/logpenparam={log_pen_param}-coef.npy'
    np.save(sm_folder + file_name_coef, coef_smpen)

    file_name_den = f'/logpenparam={log_pen_param}-densityvals-newdata.npy'
    np.save(sm_folder + file_name_den, den_vals['den_vals'])
    

## Negative Log-likelihood Density Estimation

In [ ]:
ml_folder = (f'../data/PenML-ContamData={contam_pt}-bw={bw}-' + 
             f'kernel={kernel_type}-plotdomain={xlimit}-plotcnts={plot_pts_cnt}-' + 
             f'seed={seed_no}')
if not os.path.isdir(ml_folder):
    os.mkdir(ml_folder)

file_name_newdata = f'/new_data.npy'
np.save(ml_folder + file_name_newdata, new_data)

file_name_grid_points = f'/grid_points.npy'
np.save(ml_folder + file_name_grid_points, grid_points)

In [ ]:
# Here, we choose the penalty parameter to be exp(-5.0), supplied using the lambda_param argument. 
nll_pen = NegLogLikGridPoints(
    data = waiting, 
    grid_points = grid_points, 
    base_density = base_density,
    kernel_type = 'gaussian_poly2', 
    kernel_r1 = 1.0, 
    kernel_r2 = 0.0, 
    kernel_c = 0.0, 
    kernel_bw = bw
)

In [ ]:
from datetime import datetime as dt

In [ ]:
seed_no = 1
# set parameters for computing the density estimate
bmc_params = batch_montecarlo_params(
    mc_batch_size = 5000, 
    mc_tol = 5e-3)

gdalgo_params = negloglik_optalgo_params(
    start_pt = np.zeros((grid_points.shape[0], 1)), 
    step_size = 0.4, 
    max_iter = 50, 
    rel_tol = 1e-5, 
    abs_tol = 0.05
)

ml_log_pen_param_list = np.array([-20., -19., -18., -17., -16.])
ml_log_pen_param_list

In [ ]:
waiting = waiting.reshape(-1, 1)
for log_pen_param in ml_log_pen_param_list: 
    
    print('*' * 50)
    print(f'log penalty = {log_pen_param}')
    
    np.random.seed(seed_no)
    print(f'start time = {dt.now().strftime("%H:%M:%S")}')
    nll_pen_coef = nll_pen.coef(
        data = waiting, 
        lambda_param = np.exp(log_pen_param), 
        optalgo_params = gdalgo_params, 
        batchmc_params = bmc_params, 
        batch_mc = True, 
        print_error = True
    )
    print(f'end time = {dt.now().strftime("%H:%M:%S")}')
    
    # visualize density estimate
    den_vals = plot_density_1d(
        data = waiting, 
        kernel_function = nll_pen.kernel_function, 
        base_density = base_density, 
        basis_type = 'grid_points', 
        coef = nll_pen_coef, 
        normalizing = True, 
        method = 'negative log-likelihood (smbasis)', 
        x_label = 'waiting',
        grid_points = grid_points, 
        save_plot = False, 
        save_dir = None, 
        save_filename = None, 
        plot_kwargs = plot_params)
    
    kernel_function_grid = GaussianPoly2(
        data=grid_points.reshape(-1, 1),
        r1=1.,
        r2=0.,
        c=0.,
        bw=bw)

    gram_grid = kernel_function_grid.kernel_gram_matrix(grid_points.reshape(-1, 1))

    print(np.sqrt(nll_pen_coef.T @ gram_grid @ nll_pen_coef).item())
    
    file_name_coef = f'/logpenparam={log_pen_param}-coef.npy'
    np.save(ml_folder + file_name_coef, nll_pen_coef)

    file_name_den = f'/logpenparam={log_pen_param}-densityvals-newdata.npy'
    np.save(ml_folder + file_name_den, den_vals['den_vals'])
    
    

In [ ]:
log_pen_list = np.concatenate((np.arange(-20., -9.), np.arange(-9.5, 1.5, 0.5)))
log_pen_list


In [ ]:
log_pen_list = np.concatenate((np.arange(-20., -9.), np.arange(-9.5, 1.5, 0.5)))

for log_pen in log_pen_list: 

    print(log_pen)
    file_name_den = f'/logpenparam={log_pen}-densityvals-newdata.npy'
    den_vals = np.load(ml_folder + file_name_den)
    
    plt.figure(figsize = (10, 10))
    plt.plot(new_data, den_vals, color = 'tab:blue', lw = 3.0)
    plt.hist(waiting, bins='fd', density = True, alpha = 0.3, color = 'tab:green')
    plt.axvline(x = contam_pt, color = 'k', linestyle='--')
    plt.ylim(ylimit)
    plt.show()
    
    

In [ ]:
kernel_function_grid = GaussianPoly2(
    data=grid_points.reshape(-1, 1),
    r1=1.,
    r2=0.,
    c=0.,
    bw=bw)

gram_grid = kernel_function_grid.kernel_gram_matrix(grid_points.reshape(-1, 1))

np.sqrt(nll_pen_coef.T @ gram_grid @ nll_pen_coef).item()

In [ ]:
np.random.seed(0)

nll_algo_params_cv = negloglik_optalgo_params(
    start_pt = np.zeros((waiting.shape[0], 1)), 
    step_size = [0.05, 0.02, 0.005], 
    max_iter = 20, 
    rel_tol = 3e-2, 
    abs_tol = 0.05
)

coef_nll_pen_gubasis = nll_pen_gubasis.penalized_optlambda(
    lambda_cand = np.exp(np.array([-10., -5., 0.])), 
    k_folds = 2, 
    print_error = True, 
    optalgo_params = nll_algo_params_cv, 
    batchmc_params = batchmc_params,
    save_dir = None, 
    save_info = False, 
    batch_mc=True)

In [ ]:
# specify the plotting parameters 
plot_params = plot_density_1d_params(xlimit, ylimit)

# visualize density estimate
den_vals = plot_density_1d(
    data = waiting, 
    kernel_function = kernel_function, 
    base_density = base_density, 
    basis_type = 'gubasis', 
    coef = coef_nll_pen_gubasis['opt_coef'], 
    normalizing = True, 
    method = 'negative log-likelihood (gubasis)', 
    x_label = 'waiting',
    grid_points = None,
    save_plot = False, 
    save_dir = None, 
    save_filename = None, 
    plot_kwargs = plot_params)